# 01 - Gold set, Kappa y construcción final

**Fase 2 v3 - ABSA Turismo Perú**

Este notebook valida las anotaciones de tres anotadores y construye el `gold_set_final.csv` que será usado por el modelo ABSA.

Anotadores:

| Anotador | Archivo |
|---|---|
| Álvaro | `muestra_anotador_1.csv` |
| Moisés | `muestra_anotador_2.csv` |
| Erick | `muestra_anotador_3.csv` |

**Idea metodológica:** primero se valida el acuerdo interanotador. Recién después se genera el gold set final. No se entrena el modelo con anotaciones vacías, aspectos inválidos ni desacuerdos sin resolver.

In [1]:
# ============================================================
# 1. Librerías y configuración general
# ============================================================

from pathlib import Path
from collections import Counter
import json
import warnings

import numpy as np
import pandas as pd

try:
    from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix
    from sklearn.model_selection import train_test_split
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    warnings.warn(f"scikit-learn no está disponible. Se usará implementación simple donde sea posible. Detalle: {e}")

SEED = 42
np.random.seed(SEED)

# Si el notebook se ejecuta dentro de la carpeta de notebooks, BASE_DIR será la carpeta padre.
# Si se ejecuta desde la raíz del proyecto, BASE_DIR será la carpeta actual.
# Se aceptan variantes de nombre de carpeta para que funcione aunque cambie el nombre.
NOTEBOOK_DIR_NAMES = {"notebooks", "notbooks"}
BASE_DIR = Path.cwd().parent if Path.cwd().name.lower() in NOTEBOOK_DIR_NAMES else Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
REPORTS_DIR = OUTPUT_DIR / "reports"

for folder in [DATA_DIR, OUTPUT_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: D:\Tesis\ABSA_Turismo_Fase2_V3
DATA_DIR: D:\Tesis\ABSA_Turismo_Fase2_V3\data
OUTPUT_DIR: D:\Tesis\ABSA_Turismo_Fase2_V3\outputs


## 2. Archivos de entrada

El notebook busca los archivos primero en `data/`. Si no los encuentra, también revisa la raíz del proyecto. Esto evita errores cuando estás probando localmente o en Colab/Jupyter.

In [2]:
# ============================================================
# 2. Definición de archivos de anotadores
# ============================================================

ANNOTATOR_FILES = {
    "alvaro": "muestra_anotador_1.csv",
    "moises": "muestra_anotador_2.csv",
    "erick": "muestra_anotador_3.csv",
}

REQUIRED_COLUMNS = [
    "annotation_id",
    "review_uid",
    "destination",
    "language_review",
    "text_clean",
    "aspecto",
    "input_modelo",
    "label",
]

VALID_LABELS = ["positivo", "neutro", "negativo"]
VALID_ASPECTS = [
    "atractivos",
    "costos",
    "seguridad",
    "accesibilidad",
    "limpieza",
    "atencion_servicio",
    "gastronomia",
    "alojamiento",
    "clima",
    "aforo_multitudes",
]

LABEL_NORMALIZATION = {
    "positiva": "positivo",
    "positivo": "positivo",
    "pos": "positivo",
    "+": "positivo",
    "neutral": "neutro",
    "neutra": "neutro",
    "neutro": "neutro",
    "nuetro": "neutro",
    "negativa": "negativo",
    "negativo": "negativo",
    "neg": "negativo",
    "-": "negativo",
}

ASPECT_NORMALIZATION = {
    "atención_servicio": "atencion_servicio",
    "atencion/servicio": "atencion_servicio",
    "atención / servicio": "atencion_servicio",
    "atencion servicio": "atencion_servicio",
    "aforo/multitudes": "aforo_multitudes",
    "aforo multitudes": "aforo_multitudes",
    "multitudes": "aforo_multitudes",
    "atractivo": "atractivos",
    "atractivos_turisticos": "atractivos",
}


def find_input_file(filename: str) -> Path:
    """Busca un archivo en data/ y luego en la raíz del proyecto."""
    candidates = [DATA_DIR / filename, BASE_DIR / filename, Path("/mnt/data") / filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"No se encontró {filename}. Revisa que esté en data/ o en la raíz del proyecto.")

input_paths = {name: find_input_file(filename) for name, filename in ANNOTATOR_FILES.items()}
input_paths

{'alvaro': WindowsPath('D:/Tesis/ABSA_Turismo_Fase2_V3/data/muestra_anotador_1.csv'),
 'moises': WindowsPath('D:/Tesis/ABSA_Turismo_Fase2_V3/data/muestra_anotador_2.csv'),
 'erick': WindowsPath('D:/Tesis/ABSA_Turismo_Fase2_V3/data/muestra_anotador_3.csv')}

## 3. Funciones auxiliares

Estas funciones normalizan etiquetas, validan columnas, calculan Kappa y construyen decisiones por mayoría. La normalización corrige formatos menores como mayúsculas, espacios o variantes (`positiva`, `pos`, `nuetro`). No inventa etiquetas nuevas.

In [3]:
# ============================================================
# 3. Funciones auxiliares
# ============================================================


def normalize_text_value(value):
    """Convierte valores a texto limpio. Si está vacío, devuelve NaN."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value in ["", "nan", "none", "null"]:
        return np.nan
    return value


def normalize_label(value):
    """Normaliza polaridades humanas a positivo/neutro/negativo."""
    value = normalize_text_value(value)
    if pd.isna(value):
        return np.nan
    return LABEL_NORMALIZATION.get(value, value)


def normalize_aspect(value):
    """Normaliza nombres de aspectos. Si no calza con los válidos, se conserva para reportar error."""
    value = normalize_text_value(value)
    if pd.isna(value):
        return np.nan
    return ASPECT_NORMALIZATION.get(value, value)


def validate_columns(df: pd.DataFrame, annotator_name: str):
    """Valida columnas mínimas de cada archivo."""
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"El archivo de {annotator_name} no tiene estas columnas obligatorias: {missing}")


def simple_cohen_kappa(y1, y2, labels=VALID_LABELS):
    """Implementación simple de Cohen's Kappa para fallback."""
    y1 = pd.Series(y1).reset_index(drop=True)
    y2 = pd.Series(y2).reset_index(drop=True)
    if len(y1) != len(y2) or len(y1) == 0:
        return np.nan

    observed = (y1 == y2).mean()
    p1 = y1.value_counts(normalize=True).reindex(labels, fill_value=0)
    p2 = y2.value_counts(normalize=True).reindex(labels, fill_value=0)
    expected = float((p1 * p2).sum())

    if expected == 1:
        return 1.0 if observed == 1 else np.nan
    return (observed - expected) / (1 - expected)


def fleiss_kappa_from_labels(label_matrix: pd.DataFrame, categories=VALID_LABELS):
    """
    Calcula Fleiss' Kappa para N ítems, n anotadores y k categorías.
    label_matrix debe tener una columna por anotador y solo etiquetas válidas sin NaN.
    """
    if label_matrix.empty:
        return np.nan

    counts = []
    for _, row in label_matrix.iterrows():
        counter = Counter(row.tolist())
        counts.append([counter.get(cat, 0) for cat in categories])

    M = np.array(counts, dtype=float)
    N = M.shape[0]          # número de ítems
    n = M.sum(axis=1)[0]    # anotadores por ítem, debe ser constante

    if N == 0 or n <= 1:
        return np.nan

    # Acuerdo observado por ítem
    P_i = ((M ** 2).sum(axis=1) - n) / (n * (n - 1))
    P_bar = P_i.mean()

    # Proporción total por categoría
    p_j = M.sum(axis=0) / (N * n)
    P_e = (p_j ** 2).sum()

    if P_e == 1:
        return 1.0 if P_bar == 1 else np.nan
    return (P_bar - P_e) / (1 - P_e)


def majority_vote(values, valid_values):
    """
    Devuelve valor por mayoría.
    - Si hay 3 votos iguales, devuelve ese valor y acuerdo_total.
    - Si hay 2 votos iguales, devuelve ese valor y mayoria_2_de_3.
    - Si los 3 difieren, devuelve NaN y desacuerdo_total.
    - Si hay valores vacíos o inválidos, devuelve NaN e incompleto_o_invalido.
    """
    clean = []
    for v in values:
        if pd.isna(v) or v not in valid_values:
            return np.nan, "incompleto_o_invalido"
        clean.append(v)

    counts = Counter(clean)
    value, count = counts.most_common(1)[0]

    if count == 3:
        return value, "acuerdo_total_3_de_3"
    if count == 2:
        return value, "mayoria_2_de_3"
    return np.nan, "desacuerdo_total_3_diferentes"

## 4. Carga y validación de los tres anotadores

Aquí se cargan los CSV, se normalizan polaridades/aspectos y se verifica que no existan `annotation_id` duplicados. Si un archivo tiene labels vacíos, no se inventan: se reportan para corregir o excluir.

In [4]:
# ============================================================
# 4. Carga de anotaciones
# ============================================================

annotator_dfs = {}
summary_rows = []

for name, path in input_paths.items():
    df = pd.read_csv(path, encoding="utf-8-sig")
    validate_columns(df, name)

    df = df[REQUIRED_COLUMNS].copy()
    df["annotation_id"] = df["annotation_id"].astype(str).str.strip()
    df["label"] = df["label"].apply(normalize_label)
    df["aspecto"] = df["aspecto"].apply(normalize_aspect)

    duplicate_ids = int(df["annotation_id"].duplicated().sum())
    empty_labels = int(df["label"].isna().sum())
    invalid_labels = int((~df["label"].isin(VALID_LABELS) & df["label"].notna()).sum())
    invalid_aspects = int((~df["aspecto"].isin(VALID_ASPECTS) & df["aspecto"].notna()).sum())

    if duplicate_ids > 0:
        raise ValueError(f"El archivo de {name} tiene {duplicate_ids} annotation_id duplicados. Corrige eso antes de continuar.")

    annotator_dfs[name] = df
    summary_rows.append({
        "anotador": name,
        "archivo": str(path.name),
        "filas": len(df),
        "reseñas_unicas": df["review_uid"].nunique(),
        "labels_vacios": empty_labels,
        "labels_invalidos": invalid_labels,
        "aspectos_invalidos": invalid_aspects,
        "annotation_id_duplicados": duplicate_ids,
    })

load_summary = pd.DataFrame(summary_rows)
load_summary.to_csv(REPORTS_DIR / "resumen_carga_anotadores.csv", index=False, encoding="utf-8-sig")
display(load_summary)

for name, df in annotator_dfs.items():
    print(f"\nDistribución de labels - {name}")
    display(df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="cantidad"))

,anotador,archivo,filas,reseñas_unicas,labels_vacios,labels_invalidos,aspectos_invalidos,annotation_id_duplicados
0,alvaro,muestra_anotador_1.csv,2479,1112,0,0,0,0
1,moises,muestra_anotador_2.csv,2479,1112,0,0,0,0
2,erick,muestra_anotador_3.csv,2479,1112,0,0,0,0



Distribución de labels - alvaro


,label,cantidad
0,positivo,1032
1,negativo,784
2,neutro,663



Distribución de labels - moises


,label,cantidad
0,positivo,988
1,negativo,782
2,neutro,709



Distribución de labels - erick


,label,cantidad
0,positivo,1022
1,negativo,800
2,neutro,657


## 5. Alineación por `annotation_id`

Los tres archivos deben tener las mismas filas. El orden no importa; lo que importa es que el `annotation_id` coincida. Si falta un `annotation_id` en algún anotador, se reporta.

In [5]:
# ============================================================
# 5. Alineación de anotadores por annotation_id
# ============================================================

id_sets = {name: set(df["annotation_id"]) for name, df in annotator_dfs.items()}
all_ids = set.union(*id_sets.values())
common_ids = set.intersection(*id_sets.values())

alignment_summary = []
for name, ids in id_sets.items():
    alignment_summary.append({
        "anotador": name,
        "ids_en_archivo": len(ids),
        "ids_faltantes_respecto_union": len(all_ids - ids),
        "ids_extra_respecto_interseccion": len(ids - common_ids),
    })

alignment_summary_df = pd.DataFrame(alignment_summary)
alignment_summary_df.to_csv(REPORTS_DIR / "resumen_alineacion_annotation_id.csv", index=False, encoding="utf-8-sig")
display(alignment_summary_df)

if len(common_ids) != len(all_ids):
    print("Advertencia: no todos los annotation_id están presentes en los tres archivos. Se trabajará con la intersección común.")
else:
    print("OK: los tres anotadores tienen los mismos annotation_id.")

# Base de metadatos: se toma la estructura de Álvaro como referencia, solo para columnas no anotadas.
base = annotator_dfs["alvaro"].copy()
base = base[base["annotation_id"].isin(common_ids)].copy()
base = base[["annotation_id", "review_uid", "destination", "language_review", "text_clean"]]

merged = base.copy()

for name, df in annotator_dfs.items():
    temp = df[df["annotation_id"].isin(common_ids)].copy()
    temp = temp[["annotation_id", "aspecto", "label"]].rename(columns={
        "aspecto": f"aspecto_{name}",
        "label": f"label_{name}",
    })
    merged = merged.merge(temp, on="annotation_id", how="left")

print("Filas alineadas:", merged.shape[0])
display(merged.head())

,anotador,ids_en_archivo,ids_faltantes_respecto_union,ids_extra_respecto_interseccion
0,alvaro,2479,0,0
1,moises,2479,0,0
2,erick,2479,0,0


OK: los tres anotadores tienen los mismos annotation_id.


Filas alineadas: 2479


,annotation_id,review_uid,destination,language_review,text_clean,aspecto_alvaro,label_alvaro,aspecto_moises,label_moises,aspecto_erick,label_erick
0,A001562,gm_2ddf14298befde5a,Museo de Sitio Huaca Pucllana,es,"Lugar descubierto en el año 1985, rodeado de l...",atractivos,positivo,atractivos,neutro,atractivos,neutro
1,A000002,gm_543e6a844111d4d8,Reserva Nacional de Paracas,es,"Una experiencia maravillosa, recomiendo este l...",atractivos,positivo,atractivos,positivo,atractivos,positivo
2,A001563,gm_078215b204d8e2d5,Ciudadela de Kuélap,es,"No pudimos ir, existe un divorcio entre quiene...",costos,negativo,costos,negativo,costos,negativo
3,A001564,gm_078215b204d8e2d5,Ciudadela de Kuélap,es,"No pudimos ir, existe un divorcio entre quiene...",accesibilidad,negativo,accesibilidad,negativo,accesibilidad,negativo
4,A001565,gm_be225c8c3dea6b8c,Reserva Nacional de Paracas,en,For those who've never seen a desert this is a...,atractivos,positivo,atractivos,positivo,atractivos,positivo


## 6. Validación de aspectos

Aunque la evaluación principal será sobre polaridad, el aspecto también debe ser válido. Una fila con aspecto inválido no debe pasar directamente al entrenamiento. Se envía a desacuerdo/adjudicación.

In [6]:
# ============================================================
# 6. Decisión de aspecto final por mayoría
# ============================================================

aspect_cols = ["aspecto_alvaro", "aspecto_moises", "aspecto_erick"]
label_cols = ["label_alvaro", "label_moises", "label_erick"]

aspect_decisions = merged[aspect_cols].apply(
    lambda row: majority_vote(row.tolist(), VALID_ASPECTS), axis=1
)
merged["aspecto"], merged["estado_acuerdo_aspecto"] = zip(*aspect_decisions)

aspect_summary = merged["estado_acuerdo_aspecto"].value_counts(dropna=False).rename_axis("estado_acuerdo_aspecto").reset_index(name="cantidad")
aspect_summary.to_csv(REPORTS_DIR / "resumen_acuerdo_aspectos.csv", index=False, encoding="utf-8-sig")
display(aspect_summary)

invalid_aspect_rows = merged[merged["estado_acuerdo_aspecto"] == "incompleto_o_invalido"].copy()
invalid_aspect_rows.to_csv(OUTPUT_DIR / "aspectos_invalidos_para_revision.csv", index=False, encoding="utf-8-sig")
print("Filas con aspecto incompleto o inválido:", len(invalid_aspect_rows))

,estado_acuerdo_aspecto,cantidad
0,acuerdo_total_3_de_3,2479


Filas con aspecto incompleto o inválido: 0


## 7. Kappa por pares y Fleiss Kappa global

- **Cohen's Kappa** se calcula por pares de anotadores.
- **Fleiss' Kappa** se calcula como acuerdo global entre los tres.

Solo se comparan filas con etiquetas válidas. Los labels vacíos o inválidos no se fuerzan.

In [7]:
# ============================================================
# 7. Cálculo de acuerdo interanotador
# ============================================================

pairs = [
    ("alvaro", "moises"),
    ("alvaro", "erick"),
    ("moises", "erick"),
]

pair_rows = []
for a, b in pairs:
    col_a = f"label_{a}"
    col_b = f"label_{b}"
    pair_df = merged[[col_a, col_b]].copy()
    valid_pair = pair_df[col_a].isin(VALID_LABELS) & pair_df[col_b].isin(VALID_LABELS)
    y_a = pair_df.loc[valid_pair, col_a]
    y_b = pair_df.loc[valid_pair, col_b]

    if len(y_a) == 0:
        kappa = np.nan
    elif SKLEARN_AVAILABLE:
        kappa = cohen_kappa_score(y_a, y_b, labels=VALID_LABELS)
    else:
        kappa = simple_cohen_kappa(y_a, y_b, labels=VALID_LABELS)

    agreement_pct = float((y_a.values == y_b.values).mean() * 100) if len(y_a) else np.nan

    pair_rows.append({
        "par_anotadores": f"{a}_vs_{b}",
        "n_comparado": int(len(y_a)),
        "acuerdo_porcentaje": round(agreement_pct, 2) if not pd.isna(agreement_pct) else np.nan,
        "cohen_kappa": round(float(kappa), 4) if not pd.isna(kappa) else np.nan,
    })

kappa_por_pares = pd.DataFrame(pair_rows)
kappa_por_pares.to_csv(OUTPUT_DIR / "kappa_por_pares.csv", index=False, encoding="utf-8-sig")
kappa_por_pares.to_csv(REPORTS_DIR / "kappa_por_pares.csv", index=False, encoding="utf-8-sig")
display(kappa_por_pares)

# Fleiss Kappa global: solo filas con 3 labels válidos.
valid_global = merged[label_cols].apply(lambda row: all(v in VALID_LABELS for v in row), axis=1)
label_matrix = merged.loc[valid_global, label_cols].copy()
fleiss_value = fleiss_kappa_from_labels(label_matrix, VALID_LABELS)

fleiss_kappa_global = pd.DataFrame([{
    "n_items_comparados": int(len(label_matrix)),
    "n_anotadores": 3,
    "categorias": ", ".join(VALID_LABELS),
    "fleiss_kappa": round(float(fleiss_value), 4) if not pd.isna(fleiss_value) else np.nan,
}])

fleiss_kappa_global.to_csv(OUTPUT_DIR / "fleiss_kappa_global.csv", index=False, encoding="utf-8-sig")
fleiss_kappa_global.to_csv(REPORTS_DIR / "fleiss_kappa_global.csv", index=False, encoding="utf-8-sig")
display(fleiss_kappa_global)

,par_anotadores,n_comparado,acuerdo_porcentaje,cohen_kappa
0,alvaro_vs_moises,2479,90.08,0.8492
1,alvaro_vs_erick,2479,93.63,0.9028
2,moises_vs_erick,2479,91.29,0.8676


,n_items_comparados,n_anotadores,categorias,fleiss_kappa
0,2479,3,"positivo, neutro, negativo",0.8731


In [8]:
# ============================================================
# 7.b VEREDICTO DE ACUERDO INTERANOTADOR (criterio de paso a Fase 2)
# ============================================================
# Este bloque deja EXPLÍCITO el número de Kappa, que es el criterio que
# habilita (o no) el paso a la Fase 2. Se recalcula en cada ejecución,
# así que el veredicto nunca queda desactualizado.

def interpretar_kappa(k):
    """Interpretación de Kappa según Landis & Koch (1977)."""
    if pd.isna(k):
        return "no calculable"
    if k < 0.00:
        return "sin acuerdo"
    if k <= 0.20:
        return "leve"
    if k <= 0.40:
        return "aceptable"
    if k <= 0.60:
        return "moderado"
    if k <= 0.80:
        return "sustancial"
    return "casi perfecto"

UMBRAL_FASE2 = 0.61  # mínimo "sustancial" exigido para habilitar la Fase 2

print("=" * 64)
print("VEREDICTO DE ACUERDO INTERANOTADOR")
print("=" * 64)

print("\nCohen's Kappa por pares:")
for _, r in kappa_por_pares.iterrows():
    print(f"  - {r['par_anotadores']:<18} kappa = {r['cohen_kappa']:.4f}  ({interpretar_kappa(r['cohen_kappa'])})")

print(f"\n>>> FLEISS' KAPPA GLOBAL = {fleiss_value:.4f}  ({interpretar_kappa(fleiss_value)})")
print(f"    Umbral minimo Fase 2  = {UMBRAL_FASE2:.2f}  (sustancial)")

habilitado = (not pd.isna(fleiss_value)) and fleiss_value >= UMBRAL_FASE2
if habilitado:
    print(f"\n    RESULTADO: HABILITADO PARA FASE 2   "
          f"(kappa {fleiss_value:.4f} >= {UMBRAL_FASE2:.2f})")
else:
    print(f"\n    RESULTADO: NO HABILITADO - revisar anotaciones antes de continuar   "
          f"(kappa {fleiss_value:.4f} < {UMBRAL_FASE2:.2f})")
print("=" * 64)

# Se guarda el veredicto como archivo para trazabilidad / citas en la tesis.
veredicto_fase2 = pd.DataFrame([{
    "fleiss_kappa_global": round(float(fleiss_value), 4) if not pd.isna(fleiss_value) else np.nan,
    "interpretacion": interpretar_kappa(fleiss_value),
    "umbral_fase2": UMBRAL_FASE2,
    "habilitado_fase2": bool(habilitado),
    "kappa_alvaro_moises": float(kappa_por_pares.loc[kappa_por_pares["par_anotadores"] == "alvaro_vs_moises", "cohen_kappa"].iloc[0]),
    "kappa_alvaro_erick": float(kappa_por_pares.loc[kappa_por_pares["par_anotadores"] == "alvaro_vs_erick", "cohen_kappa"].iloc[0]),
    "kappa_moises_erick": float(kappa_por_pares.loc[kappa_por_pares["par_anotadores"] == "moises_vs_erick", "cohen_kappa"].iloc[0]),
}])
veredicto_fase2.to_csv(OUTPUT_DIR / "veredicto_fase2.csv", index=False, encoding="utf-8-sig")
veredicto_fase2.to_csv(REPORTS_DIR / "veredicto_fase2.csv", index=False, encoding="utf-8-sig")
display(veredicto_fase2)

VEREDICTO DE ACUERDO INTERANOTADOR

Cohen's Kappa por pares:
  - alvaro_vs_moises   kappa = 0.8492  (casi perfecto)
  - alvaro_vs_erick    kappa = 0.9028  (casi perfecto)
  - moises_vs_erick    kappa = 0.8676  (casi perfecto)

>>> FLEISS' KAPPA GLOBAL = 0.8731  (casi perfecto)
    Umbral minimo Fase 2  = 0.61  (sustancial)

    RESULTADO: HABILITADO PARA FASE 2   (kappa 0.8731 >= 0.61)


,fleiss_kappa_global,interpretacion,umbral_fase2,habilitado_fase2,kappa_alvaro_moises,kappa_alvaro_erick,kappa_moises_erick
0,0.8731,casi perfecto,0.61,True,0.8492,0.9028,0.8676


## 8. Construcción del consenso por mayoría

Reglas:

| Caso | Acción |
|---|---|
| 3 coinciden | Se acepta como etiqueta final |
| 2 coinciden | Se acepta por mayoría |
| 3 diferentes | Se manda a desacuerdo/adjudicación |
| Label vacío o inválido | Se excluye del gold set final hasta corregir |
| Aspecto inválido | Se excluye del gold set final hasta corregir |

In [9]:
# ============================================================
# 8. Consenso de polaridad y detección de conflictos
# ============================================================

label_decisions = merged[label_cols].apply(
    lambda row: majority_vote(row.tolist(), VALID_LABELS), axis=1
)
merged["label"], merged["estado_acuerdo_label"] = zip(*label_decisions)

# Estado final de uso para entrenamiento.
# Solo entran filas con aspecto final válido y label final válido.
def decide_training_status(row):
    if row["estado_acuerdo_aspecto"] in ["incompleto_o_invalido", "desacuerdo_total_3_diferentes"]:
        return "excluir_revision_aspecto"
    if row["estado_acuerdo_label"] == "incompleto_o_invalido":
        return "excluir_label_incompleto_o_invalido"
    if row["estado_acuerdo_label"] == "desacuerdo_total_3_diferentes":
        return "excluir_desacuerdo_total_label"
    if row["label"] in VALID_LABELS and row["aspecto"] in VALID_ASPECTS:
        return "usar_en_gold_set_final"
    return "excluir_revision"

merged["estado_final"] = merged.apply(decide_training_status, axis=1)

resumen_acuerdo = merged[["estado_acuerdo_aspecto", "estado_acuerdo_label", "estado_final"]].value_counts().reset_index(name="cantidad")
resumen_acuerdo.to_csv(OUTPUT_DIR / "resumen_acuerdo_anotadores.csv", index=False, encoding="utf-8-sig")
resumen_acuerdo.to_csv(REPORTS_DIR / "resumen_acuerdo_anotadores.csv", index=False, encoding="utf-8-sig")
display(resumen_acuerdo)

print("Distribución de estado_acuerdo_label:")
display(merged["estado_acuerdo_label"].value_counts(dropna=False).rename_axis("estado").reset_index(name="cantidad"))

print("Distribución de estado_final:")
display(merged["estado_final"].value_counts(dropna=False).rename_axis("estado_final").reset_index(name="cantidad"))

,estado_acuerdo_aspecto,estado_acuerdo_label,estado_final,cantidad
0,acuerdo_total_3_de_3,acuerdo_total_3_de_3,usar_en_gold_set_final,2188
1,acuerdo_total_3_de_3,mayoria_2_de_3,usar_en_gold_set_final,253
2,acuerdo_total_3_de_3,desacuerdo_total_3_diferentes,excluir_desacuerdo_total_label,38


Distribución de estado_acuerdo_label:


,estado,cantidad
0,acuerdo_total_3_de_3,2188
1,mayoria_2_de_3,253
2,desacuerdo_total_3_diferentes,38


Distribución de estado_final:


,estado_final,cantidad
0,usar_en_gold_set_final,2441
1,excluir_desacuerdo_total_label,38


## 9. Exportación de desacuerdos y gold set final

Los desacuerdos no se destruyen. Se exportan para revisión. El gold set final queda solo con filas confiables por acuerdo total o mayoría.

In [10]:
# ============================================================
# 9. Gold set final y desacuerdos
# ============================================================

# Desacuerdos o filas no aptas para entrenamiento directo
desacuerdos = merged[merged["estado_final"] != "usar_en_gold_set_final"].copy()
desacuerdos.to_csv(OUTPUT_DIR / "desacuerdos_anotadores.csv", index=False, encoding="utf-8-sig")
desacuerdos.to_csv(REPORTS_DIR / "desacuerdos_anotadores.csv", index=False, encoding="utf-8-sig")

# Gold set final limpio
gold_set_final = merged[merged["estado_final"] == "usar_en_gold_set_final"].copy()

# input_modelo se reconstruye con el aspecto final, no con el sugerido.
gold_set_final["input_modelo"] = (
    "aspecto: " + gold_set_final["aspecto"].astype(str) +
    " reseña: " + gold_set_final["text_clean"].astype(str)
)

gold_set_final = gold_set_final[[
    "review_uid",
    "annotation_id",
    "destination",
    "language_review",
    "aspecto",
    "text_clean",
    "input_modelo",
    "label",
    "estado_acuerdo_aspecto",
    "estado_acuerdo_label",
]].copy()

# Validaciones duras antes de guardar
assert gold_set_final["label"].isin(VALID_LABELS).all(), "Hay labels inválidos en gold_set_final."
assert gold_set_final["aspecto"].isin(VALID_ASPECTS).all(), "Hay aspectos inválidos en gold_set_final."
assert gold_set_final["review_uid"].notna().all(), "Hay review_uid vacíos."
assert gold_set_final["input_modelo"].notna().all(), "Hay input_modelo vacío."

# Salidas principales
output_gold = OUTPUT_DIR / "gold_set_final.csv"
data_gold = DATA_DIR / "gold_set_final.csv"  # copia de compatibilidad para el notebook 02

gold_set_final.to_csv(output_gold, index=False, encoding="utf-8-sig")
gold_set_final.to_csv(data_gold, index=False, encoding="utf-8-sig")

print("Gold set final guardado en:", output_gold)
print("Copia en data:", data_gold)
print("Filas gold set final:", len(gold_set_final))
print("Filas enviadas a desacuerdo/revisión:", len(desacuerdos))

display(gold_set_final.head())

Gold set final guardado en: D:\Tesis\ABSA_Turismo_Fase2_V3\outputs\gold_set_final.csv
Copia en data: D:\Tesis\ABSA_Turismo_Fase2_V3\data\gold_set_final.csv
Filas gold set final: 2441
Filas enviadas a desacuerdo/revisión: 38


,review_uid,annotation_id,destination,language_review,aspecto,text_clean,input_modelo,label,estado_acuerdo_aspecto,estado_acuerdo_label
0,gm_2ddf14298befde5a,A001562,Museo de Sitio Huaca Pucllana,es,atractivos,"Lugar descubierto en el año 1985, rodeado de l...",aspecto: atractivos reseña: Lugar descubierto ...,neutro,acuerdo_total_3_de_3,mayoria_2_de_3
1,gm_543e6a844111d4d8,A000002,Reserva Nacional de Paracas,es,atractivos,"Una experiencia maravillosa, recomiendo este l...",aspecto: atractivos reseña: Una experiencia ma...,positivo,acuerdo_total_3_de_3,acuerdo_total_3_de_3
2,gm_078215b204d8e2d5,A001563,Ciudadela de Kuélap,es,costos,"No pudimos ir, existe un divorcio entre quiene...","aspecto: costos reseña: No pudimos ir, existe ...",negativo,acuerdo_total_3_de_3,acuerdo_total_3_de_3
3,gm_078215b204d8e2d5,A001564,Ciudadela de Kuélap,es,accesibilidad,"No pudimos ir, existe un divorcio entre quiene...","aspecto: accesibilidad reseña: No pudimos ir, ...",negativo,acuerdo_total_3_de_3,acuerdo_total_3_de_3
4,gm_be225c8c3dea6b8c,A001565,Reserva Nacional de Paracas,en,atractivos,For those who've never seen a desert this is a...,aspecto: atractivos reseña: For those who've n...,positivo,acuerdo_total_3_de_3,acuerdo_total_3_de_3


## 10. Distribución del gold set final

Antes de entrenar BERT + TextCNN, hay que mirar si el gold set final conserva diversidad por clase, aspecto, destino e idioma.

In [11]:
# ============================================================
# 10. Reportes de distribución del gold set final
# ============================================================

distribucion_label = gold_set_final["label"].value_counts().rename_axis("label").reset_index(name="cantidad")
distribucion_aspecto = gold_set_final["aspecto"].value_counts().rename_axis("aspecto").reset_index(name="cantidad")
distribucion_destino = gold_set_final["destination"].value_counts().rename_axis("destination").reset_index(name="cantidad")
distribucion_idioma = gold_set_final["language_review"].value_counts().rename_axis("language_review").reset_index(name="cantidad")

distribucion_label.to_csv(REPORTS_DIR / "distribucion_gold_set_por_label.csv", index=False, encoding="utf-8-sig")
distribucion_aspecto.to_csv(REPORTS_DIR / "distribucion_gold_set_por_aspecto.csv", index=False, encoding="utf-8-sig")
distribucion_destino.to_csv(REPORTS_DIR / "distribucion_gold_set_por_destino.csv", index=False, encoding="utf-8-sig")
distribucion_idioma.to_csv(REPORTS_DIR / "distribucion_gold_set_por_idioma.csv", index=False, encoding="utf-8-sig")

print("Distribución por label")
display(distribucion_label)

print("Distribución por aspecto")
display(distribucion_aspecto)

print("Distribución por idioma")
display(distribucion_idioma)

print("Distribución por destino")
display(distribucion_destino)

Distribución por label


,label,cantidad
0,positivo,1024
1,negativo,786
2,neutro,631


Distribución por aspecto


,aspecto,cantidad
0,atractivos,898
1,costos,349
2,accesibilidad,326
3,atencion_servicio,309
4,aforo_multitudes,182
5,limpieza,97
6,gastronomia,90
7,seguridad,85
8,clima,73
9,alojamiento,32


Distribución por idioma


,language_review,cantidad
0,es,1340
1,en,1101


Distribución por destino


,destination,cantidad
0,Santuario Histórico de Machu Picchu,381
1,Museo de Sitio Huaca Pucllana,320
2,Circuito Mágico del Agua,270
3,Monasterio de Santa Catalina,245
4,Ciudadela de Kuélap,232
5,Centro Histórico de Lima,226
6,Reserva Nacional de Paracas,205
7,Museo Tumbas Reales del Señor de Sipán,197
8,Líneas y Geoglifos de Nasca y Palpa,191
9,Valle del Colca,174


## 11. Split train / validation / test

El split se hace por `review_uid`, no por fila. Esto evita fuga de información: una misma reseña puede tener varias tuplas aspecto-reseña, y no debe aparecer repartida entre entrenamiento y prueba.

Proporción usada:

- 70% entrenamiento
- 15% validación
- 15% prueba

In [12]:
# ============================================================
# 11. Split por review_uid para evitar leakage
# ============================================================

SPLIT_TRAIN = 0.70
SPLIT_VALIDATION = 0.15
SPLIT_TEST = 0.15

# Una etiqueta dominante por review_uid para intentar estratificación a nivel de reseña.
uid_labels = (
    gold_set_final.groupby("review_uid")["label"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)

# .to_numpy(dtype=object) evita el conflicto entre los arrays respaldados por PyArrow
# (pandas >= 3.0) y el indexado interno de scikit-learn en train_test_split.
unique_uids = uid_labels["review_uid"].to_numpy(dtype=object)
uid_y = uid_labels["label"].to_numpy(dtype=object)

# Estratificación solo si todas las clases tienen al menos 2 grupos.
can_stratify = pd.Series(uid_y).value_counts().min() >= 2
stratify_y = uid_y if can_stratify and SKLEARN_AVAILABLE else None

if SKLEARN_AVAILABLE:
    train_uids, temp_uids, y_train, y_temp = train_test_split(
        unique_uids,
        uid_y,
        test_size=(1 - SPLIT_TRAIN),
        random_state=SEED,
        stratify=stratify_y,
    )

    temp_stratify = y_temp if pd.Series(y_temp).value_counts().min() >= 2 else None
    validation_ratio_inside_temp = SPLIT_VALIDATION / (SPLIT_VALIDATION + SPLIT_TEST)

    validation_uids, test_uids = train_test_split(
        temp_uids,
        test_size=(1 - validation_ratio_inside_temp),
        random_state=SEED,
        stratify=temp_stratify,
    )
else:
    rng = np.random.default_rng(SEED)
    shuffled = np.array(unique_uids)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * SPLIT_TRAIN)
    n_val = int(n * SPLIT_VALIDATION)
    train_uids = shuffled[:n_train]
    validation_uids = shuffled[n_train:n_train + n_val]
    test_uids = shuffled[n_train + n_val:]

train_df = gold_set_final[gold_set_final["review_uid"].isin(train_uids)].copy()
validation_df = gold_set_final[gold_set_final["review_uid"].isin(validation_uids)].copy()
test_df = gold_set_final[gold_set_final["review_uid"].isin(test_uids)].copy()

# Validación de fuga de información
assert set(train_df["review_uid"]).isdisjoint(set(validation_df["review_uid"])), "Leakage entre train y validation."
assert set(train_df["review_uid"]).isdisjoint(set(test_df["review_uid"])), "Leakage entre train y test."
assert set(validation_df["review_uid"]).isdisjoint(set(test_df["review_uid"])), "Leakage entre validation y test."

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False, encoding="utf-8-sig")
validation_df.to_csv(OUTPUT_DIR / "validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False, encoding="utf-8-sig")

# Copia en data para compatibilidad, por si el notebook 02 lee desde data/
train_df.to_csv(DATA_DIR / "train.csv", index=False, encoding="utf-8-sig")
validation_df.to_csv(DATA_DIR / "validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(DATA_DIR / "test.csv", index=False, encoding="utf-8-sig")

split_summary = pd.DataFrame([
    {"split": "train", "filas": len(train_df), "review_uid_unicos": train_df["review_uid"].nunique()},
    {"split": "validation", "filas": len(validation_df), "review_uid_unicos": validation_df["review_uid"].nunique()},
    {"split": "test", "filas": len(test_df), "review_uid_unicos": test_df["review_uid"].nunique()},
])
split_summary.to_csv(REPORTS_DIR / "resumen_splits_gold_set.csv", index=False, encoding="utf-8-sig")
display(split_summary)

print("Distribución por label en train")
display(train_df["label"].value_counts().rename_axis("label").reset_index(name="cantidad"))
print("Distribución por label en validation")
display(validation_df["label"].value_counts().rename_axis("label").reset_index(name="cantidad"))
print("Distribución por label en test")
display(test_df["label"].value_counts().rename_axis("label").reset_index(name="cantidad"))

,split,filas,review_uid_unicos
0,train,1679,773
1,validation,365,166
2,test,397,166


Distribución por label en train


,label,cantidad
0,positivo,703
1,negativo,544
2,neutro,432


Distribución por label en validation


,label,cantidad
0,positivo,161
1,negativo,113
2,neutro,91


Distribución por label en test


,label,cantidad
0,positivo,160
1,negativo,129
2,neutro,108


## 12. Resumen final de archivos generados

Este bloque deja un registro claro de qué archivos produjo el notebook. Eso te sirve para trazabilidad y para documentarlo en la tesis.

In [13]:
# ============================================================
# 12. Registro final de salidas
# ============================================================

outputs_generated = [
    OUTPUT_DIR / "kappa_por_pares.csv",
    OUTPUT_DIR / "fleiss_kappa_global.csv",
    OUTPUT_DIR / "resumen_acuerdo_anotadores.csv",
    OUTPUT_DIR / "desacuerdos_anotadores.csv",
    OUTPUT_DIR / "aspectos_invalidos_para_revision.csv",
    OUTPUT_DIR / "gold_set_final.csv",
    OUTPUT_DIR / "train.csv",
    OUTPUT_DIR / "validation.csv",
    OUTPUT_DIR / "test.csv",
    DATA_DIR / "gold_set_final.csv",
    DATA_DIR / "train.csv",
    DATA_DIR / "validation.csv",
    DATA_DIR / "test.csv",
]

files_report = pd.DataFrame([
    {
        "archivo": str(p.relative_to(BASE_DIR)) if str(p).startswith(str(BASE_DIR)) else str(p),
        "existe": p.exists(),
        "tamaño_bytes": p.stat().st_size if p.exists() else 0,
    }
    for p in outputs_generated
])

files_report.to_csv(OUTPUT_DIR / "archivos_generados_notebook_01_fase2.csv", index=False, encoding="utf-8-sig")
display(files_report)

execution_log = {
    "notebook": "01_gold_set_kappa_y_construccion_final.ipynb",
    "anotadores": list(ANNOTATOR_FILES.keys()),
    "filas_alineadas": int(len(merged)),
    "filas_gold_set_final": int(len(gold_set_final)),
    "filas_desacuerdo_revision": int(len(desacuerdos)),
    "labels_validos": VALID_LABELS,
    "aspectos_validos": VALID_ASPECTS,
    "split": {
        "train_filas": int(len(train_df)),
        "validation_filas": int(len(validation_df)),
        "test_filas": int(len(test_df)),
    },
}

with open(OUTPUT_DIR / "log_notebook_01_fase2.json", "w", encoding="utf-8") as f:
    json.dump(execution_log, f, ensure_ascii=False, indent=2)

print("Notebook 01 finalizado.")
print("Siguiente paso: ejecutar 02_absa_bert_textcnn_inferencia_matriz.ipynb usando data/gold_set_final.csv o outputs/gold_set_final.csv.")

,archivo,existe,tamaño_bytes
0,outputs\kappa_por_pares.csv,True,168
1,outputs\fleiss_kappa_global.csv,True,104
2,outputs\resumen_acuerdo_anotadores.csv,True,291
3,outputs\desacuerdos_anotadores.csv,True,18711
4,outputs\aspectos_invalidos_para_revision.csv,True,181
5,outputs\gold_set_final.csv,True,2228708
6,outputs\train.csv,True,1542398
7,outputs\validation.csv,True,299400
8,outputs\test.csv,True,387188
9,data\gold_set_final.csv,True,2228708


Notebook 01 finalizado.
Siguiente paso: ejecutar 02_absa_bert_textcnn_inferencia_matriz.ipynb usando data/gold_set_final.csv o outputs/gold_set_final.csv.
